主要实现功能：根据历史销售数据和月度系数生成预测数据

In [ ]:
import pantab as pt
import pandas as pd
from datetime import datetime

pd.options.mode.copy_on_write = True

In [ ]:
hfile = f'../src_data/hyper/退货率带to_25-12-21_tag_size_review.hyper'

parm_month_path = '../src_data/月度系数/月度系数.xlsx'
sales_program_path = '../src_data/Listing销占比程序表-模板.xlsx'

#+ 1、构建带本月系数的销占比数据

In [ ]:
# 读取“Listing销占比程序表-模板”表
sales_program = pd.read_excel(sales_program_path)

# 读取“月度系数”，parse_dates参数用于指定列被解析为日期时间类型。
parm_month = pd.read_excel(parm_month_path, parse_dates=['月份'])
parm_month['年月'] = parm_month['月份'].dt.to_period('M').astype(str)

# 获取当前月份的第一天
current_month_start = datetime(datetime.now().year, datetime.now().month, 1)

# 筛选出日期大于等于当前月份的数据 (也就是获取当前月份及未来月份的月度系数 数据)
parm_month = parm_month[parm_month['月份'] >= current_month_start]
parm_month = parm_month.sort_values(by=['Listing', '月份'], ascending=[True, True])

# 根据'店铺-站点', 'Listing'去除重复行，最终只保留当前月份的月度系数 数据
temp_parm = parm_month.drop_duplicates(subset=['店铺-站点', 'Listing'])

# 构建出带本月系数的销占比数据（将月度系数与销占比程序表合并，销占比程序表为主表）
prepaer_df = pd.merge(left=sales_program, right=temp_parm, how='left', on=['店铺-站点', 'Listing', '站点'])

In [ ]:
# parm_month.to_excel("../file/parm_month2.xlsx", index = False)

In [ ]:
parm_month = parm_month.sort_values(by=['Listing', '月份'], ascending=[True, True])

In [ ]:
parm_month

In [ ]:
parm_month['年月'].head(1)

In [ ]:
temp_parm

In [ ]:
current_month_start

In [ ]:
print("销占比程序表总行数：", len(sales_program), '\n月度系数表总行数：', len(temp_parm), '\n合并后总行数：', len(prepaer_df))

In [ ]:
prepaer_df.head()

In [ ]:
# prepaer_df.to_excel("../file/销占比与当月月度系数合并结果.xlsx", index = False)

# 2、处理历史销量

In [ ]:
sales_df = pt.frame_from_hyper(hfile, table="return_review_data")

In [ ]:
# 获取指定维度数据
sales_df = sales_df[['purchaseDate', '销量', 'Msku', '店铺-站点', 'SKU']]
sales_df.rename(columns={'Msku': 'MSKU'}, inplace=True)

# 删除空值
sales_df.dropna(subset=['销量'], inplace=True)

In [ ]:
# 获取指定店铺数据加快处理速度
# TODO 此处可以修改
sales_df = sales_df[sales_df['店铺-站点'].isin(['CIKERWEL:US','SEEKWAY:US', 'rivbos:US', 'RIVMOUNT:US','SEEKWAY:CA','RIVMOUNT:CA'])]

# 3、分别获取近N天销量和近15天销量

In [ ]:
# TODO 输入参数
year = input('请输入年份：')
month = input('请输入月份：')
days = input('请输入日期：')
sales_df['now'] = f'{year}-{month}-{days}'

# 获取日期格式
sales_df['now'] = pd.to_datetime(sales_df['now'])
sales_df['年月'] =  sales_df['now'].dt.to_period('M').astype(str)
sales_df['purchaseDate'] = pd.to_datetime(sales_df['purchaseDate'])

In [ ]:
# 销量数据：获取三个时间：订单购买时间与现在时间的差（天数）、本月总天数、本月度过的天数

# 当前时间到订单购买时间天数差
sales_df['days_diff'] = (sales_df['now'] - sales_df['purchaseDate']).dt.days
# 获取指定日期所在月天数.
# TODO 控制近期销量（获取本月的总天数）
sales_df['days_in_month'] = sales_df['now'].apply(lambda x: x.days_in_month)

# 获取本月度过的天数（假设今天是4月22日，那就是度过了21天）
sales_df['curr_day']= sales_df['now'].dt.day - 1

In [ ]:
sales_df['days_diff'].head()

In [ ]:
sales_df['days_in_month'].head()

In [ ]:
sales_df['curr_day'].head()

In [ ]:
sales_df.tail()

## 3.1 获取近N天销量
## 3.2 获取近15天销量

In [ ]:
# def calculate_sales_by_month(sales_df, separate_days):
#     """
#     计算最近N天内每个店铺每种商品的本月销量和上月销量。
    
#     :param sales_df: 销售数据的DataFrame
#     :param separate_days: 分割天数的阈值(15天/N天)
#     :return: 包含本月销量和上月销量的DataFrame
#     """
    
#     # 确保days_diff列没有异常值，这里假设days_diff是日期差，应为非负数
#     if not all(sales_df['days_diff'] >= 0):
#         raise ValueError("days_diff列包含负值或异常值")
    
#     # 分割天数过滤 (获取最近一个月或者最近15天的销售数据)
#     recently_n_sales = sales_df.loc[sales_df['days_diff'] <= separate_days]
#     # print("天数差：",recently_n_sales['days_diff'].unique())
#     # print("本月度过的天数：",recently_n_sales['curr_day'].unique())

#     # print(recently_n_sales.head())
    
#     # 边界条件检查
#     if recently_n_sales.empty:
#         raise ValueError("没有符合条件的销售数据")

#     # 进一步将 recently_n_sales 分为两部分
#     # 计算本月销量和上月销量
#     current_month_sales_n = recently_n_sales.loc[recently_n_sales['days_diff'] <= recently_n_sales['curr_day']]
#     # print("本月销量:")
#     # print("\t天数差：",current_month_sales_n['days_diff'].unique())
#     # print("\t本月度过的天数：",current_month_sales_n['curr_day'].unique())
    
#     last_month_sales_n = recently_n_sales.loc[recently_n_sales['days_diff'] > recently_n_sales['curr_day']]
#     # print("上月销量:")
#     # print("\t天数差：",last_month_sales_n['days_diff'].unique())
#     # print("\t本月度过的天数：",last_month_sales_n['curr_day'].unique())

#     print("\n")
#     # 使用函数简化重复逻辑
#     def pivot_sales(df, column_name):
#         return df.pivot_table(index=['店铺-站点', 'MSKU'], values='销量', aggfunc='sum', fill_value=0).reset_index().rename(columns={'销量': column_name})
    
#     if current_month_sales_n.empty:
#         current_month_sales_n = pd.DataFrame({'店铺-站点': [], 'MSKU': [], '本月销量': []})
#     else:
#         current_month_sales_n = pivot_sales(current_month_sales_n, '本月销量')
#     # print("本月销量\n", current_month_sales_n.head())
        
#     if last_month_sales_n.empty:
#         last_month_sales_n = pd.DataFrame({'店铺-站点': [], 'MSKU': [], '上月销量': []})
#     else:
#         last_month_sales_n = pivot_sales(last_month_sales_n, '上月销量')
#     # print("上月销量\n",last_month_sales_n.head())
        
#     # print("\n\n\n")
#     return current_month_sales_n, last_month_sales_n

In [ ]:
sales_df[sales_df['days_diff'] < 0]

In [ ]:
days_in_month = sales_df.iloc[0]['days_in_month']

In [ ]:
days_in_month

In [ ]:
def calculate_sales_by_month(sales_df, separate_days):
    """
    计算最近N天内每个店铺每种商品的本月销量和上月销量。
    
    :param sales_df: 销售数据的DataFrame
    :param separate_days: 分割天数的阈值
    :return: 包含本月销量和上月销量的DataFrame
    """
    
    # 确保days_diff列没有异常值，这里假设days_diff是日期差，应为非负数
    if not all(sales_df['days_diff'] >= 0):
        raise ValueError("days_diff列包含负值或异常值")
    
    # 分割天数过滤 (获取最近一个月或者最近15天的销售数据) separate_days为30（28，29，30，31）天或15天
    recently_n_sales = sales_df.loc[sales_df['days_diff'] <= separate_days]
    # print("天数差：",recently_n_sales['days_diff'].unique())
    # print("本月度过的天数：",recently_n_sales['curr_day'].unique())

    # print(recently_n_sales.head())
    
    # 边界条件检查
    if recently_n_sales.empty:
        raise ValueError("没有符合条件的销售数据")

    # 进一步将 recently_n_sales 分为两部分
    # 计算本月销量和上月销量
    current_month_sales_n = recently_n_sales.loc[recently_n_sales['days_diff'] <= recently_n_sales['curr_day']]
    last_month_sales_n = recently_n_sales.loc[recently_n_sales['days_diff'] > recently_n_sales['curr_day']]
    
    # 使用函数简化重复逻辑
    def pivot_sales(df, column_name):
        return df.pivot_table(index=['店铺-站点', 'MSKU'], values='销量', aggfunc='sum', fill_value=0).reset_index().rename(columns={'销量': column_name})
    
    if current_month_sales_n.empty:
        current_month_sales_n = pd.DataFrame({'店铺-站点': [], 'MSKU': [], '本月销量': []})
    else:
        current_month_sales_n = pivot_sales(current_month_sales_n, '本月销量')
        
    if last_month_sales_n.empty:
        last_month_sales_n = pd.DataFrame({'店铺-站点': [], 'MSKU': [], '上月销量': []})
    else:
        last_month_sales_n = pivot_sales(last_month_sales_n, '上月销量')
    
    return current_month_sales_n, last_month_sales_n

In [ ]:
# 获取本月总天数
# 计算最多近一个31天销量 days_in_month 取值区间为[28, 29, 30, 31]
days_in_month = sales_df.iloc[0]['days_in_month']   # 本月总天数。iloc[行，列]  iloc[:, :]
parms_day = [days_in_month, 15]
file_names = ['30', '15']


for day, file_name in zip(parms_day, file_names):
    current_month_sales_n, last_month_sales_n = calculate_sales_by_month(sales_df=sales_df, separate_days=day)

    # 在ret_df_n中添加本月销量和上月销量
    ret_df_n = pd.merge(left=prepaer_df, right=current_month_sales_n, how='left', on=['店铺-站点', 'MSKU'])
    ret_df_n['本月销量'] = ret_df_n['本月销量'].fillna(0)

    ret_df_n = pd.merge(left=ret_df_n, right=last_month_sales_n, how='left', on=['店铺-站点', 'MSKU'])
    ret_df_n['上月销量'] = ret_df_n['上月销量'].fillna(0)

    # 计算MSKU预估销量
    month_parms = 2 if day == 15 else 1
    ret_df_n['预估销量'] = (ret_df_n['上月销量'] + ret_df_n['本月销量'] / ret_df_n['系数']) * ret_df_n['系数'] * month_parms

    # 获取款式销量和Listing销量为计算销占比做准备
    style_sales_n = ret_df_n.pivot_table(index=['店铺-站点', 'Listing','款式'], values=['预估销量'], aggfunc=sum).rename(columns={'预估销量': '款式销量'})
    listing_sales_n = ret_df_n.pivot_table(index=['店铺-站点', 'Listing'], values=['预估销量'], aggfunc=sum).rename(columns={'预估销量': 'Listing销量'})

    # 匹配款式销量
    ret_zhanbi_parms = pd.merge(left=ret_df_n, right=style_sales_n, how='left', left_on=['店铺-站点', 'Listing','款式'], right_index=True)
    # 匹配Listing销量
    ret_zhanbi_parms = pd.merge(left=ret_zhanbi_parms, right=listing_sales_n, how='left', left_on=['店铺-站点', 'Listing'], right_index=True)

    # 计算销占比
    ret_zhanbi_parms['SKU销占比']= (ret_zhanbi_parms['预估销量'] / ret_zhanbi_parms['款式销量']).fillna(0)


    ret_zhanbi_parms['款式销占比']= (ret_zhanbi_parms['款式销量'] / ret_zhanbi_parms['Listing销量']).fillna(0)


    # 构建月度销量
    listing_start_sales = ret_df_n.pivot_table(index=['站点', 'Listing', '月份'], values=['预估销量'], aggfunc='sum')
    # print(listing_start_sales)
    # listing_start_sales.to_excel("../file/listing_start_sales1.xlsx", index = False)
    
    # 匹配月度系数
    listing_start_sales = pd.merge(left=parm_month, right=listing_start_sales, left_on=['站点', 'Listing', '月份'], right_index=True, how='left').reset_index()
    # print(listing_start_sales)
    # listing_start_sales.to_excel("../file/listing_start_sales2.xlsx", index = False)
    # print(len(listing_start_sales))

    # 这段代码就是一个**“向下填充”**的逻辑，不过它并不是简单地复制上一行的值，而是把上一行的“预估销量”乘以本行的一个“系数”，作为当前行的“预估销量”。
    for index in range(1, len(listing_start_sales)):
        if pd.isna(listing_start_sales.loc[index, '预估销量']):
            listing_start_sales.loc[index, '预估销量'] = listing_start_sales.loc[index - 1, '预估销量'] * listing_start_sales.loc[index, '系数']

        
    
    ret_zhanbi_parms.to_excel(f'../src_data/Listing销占比程序表-{year}{month}{days}-{file_name}.xlsx', index=False, columns=sales_program.columns)

    listing_start_sales.rename(columns={'预估销量': 'Listing_月度预估销量'}, inplace=True)
    listing_start_sales.to_excel(f'../src_data/Listing预估表-{year}{month}{days}-{file_name}.xlsx', index=False, columns=['站点', 'Listing', '月份', 'Listing_月度预估销量'])


In [ ]:
ret_df_n

In [ ]:
# ret_df_n.to_excel("../file/generateParms-ret_df_n.xlsx", index = False)